# Pipeline de Modelagem de Risco de Crédito 

**Requisito único:** `dados_credito.csv` na mesma pasta do notebook (no Colab, faça upload para `/content`).
O módulo auxiliar `feature_engineering.py` é criado automaticamente na Etapa 3. Use **Executar tudo**.

## ETAPA 0 — CONFIGURAÇÃO

In [ ]:

from pathlib import Path
import json
import os
import sys
import warnings
import platform
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
np.random.seed(SEED)

# Ajuste manualmente se o CSV estiver em outro local.
CAMINHO_ENTRADA_CSV = None  # Exemplo: r"C:\dados\dados_credito.csv"

PASTA_TRABALHO = Path.cwd()
PASTA_BASE = PASTA_TRABALHO / "psi_pd_v2"
PASTA_ARTEFATOS = PASTA_BASE / "artifacts"
PASTA_RELATORIOS = PASTA_BASE / "reports"
for pasta in (PASTA_BASE, PASTA_ARTEFATOS, PASTA_RELATORIOS):
    pasta.mkdir(parents=True, exist_ok=True)


def localizar_csv(caminho_manual=None):
    candidatos = []
    if caminho_manual:
        candidatos.append(Path(caminho_manual).expanduser())
    candidatos.extend([
        Path.cwd() / "dados_credito.csv",
        Path("/mnt/data/dados_credito.csv"),
        Path("/content/dados_credito.csv"),
    ])
    for candidato in candidatos:
        if candidato.is_file():
            return candidato.resolve()
    return None

CAMINHO_ENTRADA_CSV = localizar_csv(CAMINHO_ENTRADA_CSV)
if CAMINHO_ENTRADA_CSV is None:
    raise FileNotFoundError(
        "dados_credito.csv não encontrado. Coloque o arquivo na pasta do notebook "
        "(no Colab: /content) ou ajuste CAMINHO_ENTRADA_CSV."
    )

ID_COL = "cliente"
TARGET_COL = "marcacao_default"
NUMERIC_COLS = [
    "idade",
    "tempo_conta_bancaria",
    "renda_mensal",
    "qtde_outros_emprestimos_atual",
    "valor_mensal_prestacoes_outros_emprestimos_atual",
    "taxa_uso_credito",
    "consultas_credito_ultimos_6meses",
    "pagamentos_atrasados_ultimos_12meses",
    "valor_emprestimo_solicitado",
    "prazo_emprestimo_solicitado",
]
CATEGORICAL_COLS = [
    "genero",
    "estado_civil",
    "tipo_cidade",
    "estado",
    "tipo_emprego",
    "objetivo_emprestimo_solicitado",
    "existencia_garantia",
]
REQUIRED_COLS = [ID_COL, *NUMERIC_COLS, *CATEGORICAL_COLS, TARGET_COL]

print(f"CSV: {CAMINHO_ENTRADA_CSV}")
print(f"Saídas: {PASTA_BASE.resolve()}")
print(f"Semente aleatória: {SEED}")

## ETAPA 1 — INGESTÃO E VALIDAÇÃO DOS DADOS

In [ ]:
import re
import unicodedata


def normalizar_nome_coluna(nome):
    texto = unicodedata.normalize("NFKD", str(nome))
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    return re.sub(r"[^0-9a-zA-Z]+", "_", texto.lower()).strip("_")


def ingerir_dados(caminho):
    # O arquivo fornecido usa CSV com separador vírgula. A leitura mantém os nomes
    # originais apenas até a normalização; a validação abaixo acusa divergências.
    df = pd.read_csv(caminho, encoding="utf-8-sig")
    df.columns = [normalizar_nome_coluna(c) for c in df.columns]

    if df.columns.duplicated().any():
        duplicadas = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"Há nomes de colunas duplicados após normalização: {duplicadas}")

    faltantes = sorted(set(REQUIRED_COLS) - set(df.columns))
    extras = sorted(set(df.columns) - set(REQUIRED_COLS))
    if faltantes:
        raise ValueError(f"Contrato de dados inválido. Colunas obrigatórias ausentes: {faltantes}")

    df = df[REQUIRED_COLS].copy()
    df[ID_COL] = df[ID_COL].astype("string").str.strip().replace("", pd.NA)
    if df[ID_COL].isna().any():
        raise ValueError(f"A coluna identificadora {ID_COL} contém valores ausentes.")

    for coluna in NUMERIC_COLS + [TARGET_COL]:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    # O alvo é obrigatório, binário e não pode conter ausências.
    if df[TARGET_COL].isna().any():
        raise ValueError("A variável-alvo contém valores ausentes ou não numéricos.")
    valores_alvo = set(df[TARGET_COL].unique())
    if not valores_alvo.issubset({0, 1}):
        raise ValueError(f"A variável-alvo deve conter somente 0 e 1. Encontrado: {valores_alvo}")
    df[TARGET_COL] = df[TARGET_COL].astype("int8")

    for coluna in CATEGORICAL_COLS:
        serie = df[coluna].astype("string").str.strip().replace("", pd.NA)
        df[coluna] = serie.astype(object).where(serie.notna(), np.nan)

    relatorio_ingestao = {
        "arquivo_origem": str(caminho),
        "linhas": int(len(df)),
        "colunas": int(df.shape[1]),
        "colunas_extras_ignoradas": extras,
        "ids_duplicados": int(df[ID_COL].duplicated().sum()),
        "taxa_default": float(df[TARGET_COL].mean()),
    }
    return df, relatorio_ingestao


df_ingested, relatorio_ingestao = ingerir_dados(CAMINHO_ENTRADA_CSV)
print(f"Base carregada: {df_ingested.shape[0]:,} registros × {df_ingested.shape[1]} colunas")
print(f"Taxa de default: {df_ingested[TARGET_COL].mean():.2%}")
print(f"IDs duplicados: {relatorio_ingestao['ids_duplicados']}")
print(f"Colunas extras ignoradas: {relatorio_ingestao['colunas_extras_ignoradas']}")
display(df_ingested.head())

## ETAPA 2A — ANÁLISE EXPLORATÓRIA E DIAGNÓSTICO

In [ ]:

from sklearn.model_selection import train_test_split

X_full = df_ingested.drop(columns=[TARGET_COL]).copy()
y_full = df_ingested[TARGET_COL].copy()
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full,
    y_full,
    test_size=0.25,
    random_state=SEED,
    stratify=y_full,
)

# Estatísticas gerais e partição.
resumo_estrutura = pd.DataFrame({
    "indicador": ["Registros totais", "Variáveis", "Treino", "Teste", "Defaults totais", "Taxa de default total"],
    "valor": [
        len(df_ingested), len(df_ingested.columns) - 1,
        len(X_train_raw), len(X_test_raw), int(y_full.sum()), float(y_full.mean())
    ],
})
resumo_nulos = (
    df_ingested.isna().sum().rename("quantidade")
    .to_frame().assign(percentual=lambda x: 100 * x["quantidade"] / len(df_ingested))
    .query("quantidade > 0").sort_values("percentual", ascending=False)
)
resumo_numerico_treino = X_train_raw[NUMERIC_COLS].describe().T

# Diagnóstico de 12 regras. Valores fora do esperado são sinalizados, não removidos.
LIMITE_COMPROMETIMENTO = 0.50
prestacao = df_ingested["valor_mensal_prestacoes_outros_emprestimos_atual"]
renda = df_ingested["renda_mensal"]
regras = {
    "Prestação positiva sem empréstimo declarado": (df_ingested["qtde_outros_emprestimos_atual"] == 0) & (prestacao > 0),
    "Comprometimento atual superior a 50%": (renda > 0) & (prestacao > LIMITE_COMPROMETIMENTO * renda),
    "Idade fora do intervalo 18–100": (df_ingested["idade"] < 18) | (df_ingested["idade"] > 100),
    "Renda mensal não positiva": renda <= 0,
    "Tempo de conta negativo": df_ingested["tempo_conta_bancaria"] < 0,
    "Quantidade de empréstimos negativa": df_ingested["qtde_outros_emprestimos_atual"] < 0,
    "Prestações mensais negativas": prestacao < 0,
    "Utilização de crédito fora de 0–100": (df_ingested["taxa_uso_credito"] < 0) | (df_ingested["taxa_uso_credito"] > 100),
    "Consultas de crédito negativas": df_ingested["consultas_credito_ultimos_6meses"] < 0,
    "Quantidade de atrasos negativa": df_ingested["pagamentos_atrasados_ultimos_12meses"] < 0,
    "Valor solicitado não positivo": df_ingested["valor_emprestimo_solicitado"] <= 0,
    "Prazo solicitado não positivo": df_ingested["prazo_emprestimo_solicitado"] <= 0,
}
linhas_regras = []
for nome, mascara in regras.items():
    mascara = mascara.fillna(False)
    linhas_regras.append({
        "regra": nome,
        "registros_sinalizados": int(mascara.sum()),
        "percentual_base": float(mascara.mean() * 100),
    })
relatorio_regras = pd.DataFrame(linhas_regras).sort_values("percentual_base", ascending=False)

# Indicadores de default por comportamento, calculados uma única vez na partição treino.
df_grupos_treino = X_train_raw.copy()
df_grupos_treino[TARGET_COL] = y_train

indice_default_atrasos = (
    df_grupos_treino.groupby("pagamentos_atrasados_ultimos_12meses", observed=True)[TARGET_COL]
    .agg(registros="size", defaults="sum", taxa_default="mean")
    .reset_index()
)
faixas_uso = [0, 20, 40, 60, 80, 100]
rotulos_uso = ["0–20", "20–40", "40–60", "60–80", "80–100"]
df_grupos_treino["faixa_uso_credito"] = pd.cut(
    df_grupos_treino["taxa_uso_credito"], bins=faixas_uso, labels=rotulos_uso, include_lowest=True
)
indice_default_uso = (
    df_grupos_treino.groupby("faixa_uso_credito", observed=False)[TARGET_COL]
    .agg(registros="size", defaults="sum", taxa_default="mean")
    .reset_index()
)
# Default por categoria (treino). A amplitude mostra o quanto cada variável separa bons e maus pagadores.
indice_default_categorias = pd.concat([
    df_grupos_treino.groupby(coluna)[TARGET_COL]
    .agg(registros="size", taxa_default="mean")
    .reset_index()
    .rename(columns={coluna: "categoria"})
    .assign(variavel=coluna)
    for coluna in CATEGORICAL_COLS
])
amplitude_categorias = (
    indice_default_categorias.groupby("variavel")["taxa_default"]
    .agg(lambda t: (t.max() - t.min()) * 100)
    .rename("amplitude_pp")
    .sort_values(ascending=False)
    .reset_index()
)

# Pearson: variáveis numéricas originais + alvo binário; ID e categóricas ficam de fora.
dados_correlacao = X_train_raw[NUMERIC_COLS].copy()
dados_correlacao[TARGET_COL] = y_train
matriz_correlacao = dados_correlacao.corr(method="pearson")

print("=== Estrutura e partição ===")
display(resumo_estrutura)
print("=== Valores ausentes ===")
display(resumo_nulos)
print("=== Regras de diagnóstico ===")
display(relatorio_regras)
print("=== Default por quantidade de atrasos ===")
display(indice_default_atrasos)
print("=== Default por faixa de utilização de crédito ===")
display(indice_default_uso)
print("=== Correlações com o alvo (treino) ===")
display(matriz_correlacao[[TARGET_COL]].sort_values(TARGET_COL, ascending=False))
print("=== Amplitude da taxa de default por variável categórica (p.p.) ===")
display(amplitude_categorias.round(2))

## ETAPA 2B — VISUALIZAÇÃO DA EDA

In [ ]:
import matplotlib.pyplot as plt

# 1. Distribuição da variável-alvo
plt.figure(figsize=(6, 4))
contagem_alvo = y_full.value_counts().sort_index()
plt.bar(["Não default (0)", "Default (1)"], contagem_alvo.values, color=["#1f77b4", "#ff7f0e"])
plt.title("Distribuição da variável-alvo")
plt.ylabel("Registros")
for i, valor in enumerate(contagem_alvo.values):
    plt.text(i, valor, f"{valor:,}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(PASTA_RELATORIOS / "eda_distribuicao_alvo.png", dpi=160, bbox_inches="tight")
plt.show()

# 2. Taxa de default por quantidade de atrasos
plt.figure(figsize=(7, 4))
plt.bar(
    indice_default_atrasos["pagamentos_atrasados_ultimos_12meses"].astype(str),
    indice_default_atrasos["taxa_default"] * 100,
)
plt.title("Taxa de default por quantidade de atrasos")
plt.xlabel("Atrasos nos últimos 12 meses")
plt.ylabel("Default observado (%)")
plt.tight_layout()
plt.savefig(PASTA_RELATORIOS / "eda_default_atrasos.png", dpi=160, bbox_inches="tight")
plt.show()

# 3. Taxa de default por faixa de utilização
plt.figure(figsize=(7, 4))
plt.bar(
    indice_default_uso["faixa_uso_credito"].astype(str),
    indice_default_uso["taxa_default"] * 100,
)
plt.title("Taxa de default por utilização de crédito")
plt.xlabel("Utilização do limite (%)")
plt.ylabel("Default observado (%)")
plt.tight_layout()
plt.savefig(PASTA_RELATORIOS / "eda_default_utilizacao.png", dpi=160, bbox_inches="tight")
plt.show()

# 4. Matriz de Correlação de Pearson
fig, ax = plt.subplots(figsize=(8, 6))
matriz = matriz_correlacao.to_numpy()
im = ax.imshow(matriz, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_title("Matriz de correlação — treino")
ax.set_xticks(range(len(matriz_correlacao.columns)))
ax.set_yticks(range(len(matriz_correlacao.index)))
ax.set_xticklabels(matriz_correlacao.columns, rotation=90, fontsize=8)
ax.set_yticklabels(matriz_correlacao.index, fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(PASTA_RELATORIOS / "eda_matriz_correlacao.png", dpi=160, bbox_inches="tight")
plt.show()

# 5. Poder de separação das categóricas
plt.figure(figsize=(8, 4))
plt.barh(amplitude_categorias["variavel"], amplitude_categorias["amplitude_pp"])
plt.gca().invert_yaxis()
plt.title("Diferença entre a maior e a menor taxa de default por variável")
plt.xlabel("Amplitude (pontos percentuais)")
plt.tight_layout()
plt.savefig(PASTA_RELATORIOS / "eda_default_categoricas.png", dpi=160, bbox_inches="tight")
plt.show()


## ETAPA 3 — PREPARAÇÃO E PRÉ-PROCESSAMENTO

In [ ]:
import importlib
import sys
from pathlib import Path

# O módulo auxiliar é gerado pelo próprio notebook a cada execução (sempre sobrescrito),
# garantindo que o código e o modelo salvo usem a mesma engenharia de atributos.
# Ele fica em arquivo separado para que o joblib consiga recarregar o modelo em outro ambiente.
codigo_modulo = '''import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

DERIVED_FEATURES = [
    "pct_comprometimento_renda_atual",
    "pct_comprometimento_renda_futuro",
    "pct_solicitado_sobre_renda_anual",
    "flag_prestacao_sem_emprestimo",
]


class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    """Cria as variáveis derivadas a partir dos dados brutos. Não aprende parâmetros."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        renda = df["renda_mensal"].where(df["renda_mensal"] > 0)  # renda não positiva vira ausente
        prestacao = df["valor_mensal_prestacoes_outros_emprestimos_atual"]
        prazo = df["prazo_emprestimo_solicitado"].where(df["prazo_emprestimo_solicitado"] > 0)
        parcela_estimada = df["valor_emprestimo_solicitado"] / prazo  # principal / prazo, sem juros

        df["pct_comprometimento_renda_atual"] = 100 * prestacao / renda
        df["pct_comprometimento_renda_futuro"] = 100 * (prestacao + parcela_estimada) / renda
        df["pct_solicitado_sobre_renda_anual"] = 100 * df["valor_emprestimo_solicitado"] / (12 * renda)
        df["flag_prestacao_sem_emprestimo"] = (
            (df["qtde_outros_emprestimos_atual"] == 0) & (prestacao > 0)
        ).astype(float)
        return df
'''

caminho_modulo = Path.cwd() / "feature_engineering.py"
caminho_modulo.write_text(codigo_modulo, encoding="utf-8")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import feature_engineering
importlib.reload(feature_engineering)  # usa a versão recém-gravada mesmo ao reexecutar a célula
print(f"Módulo auxiliar gerado: {caminho_modulo}")


from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from feature_engineering import FeatureEngineeringTransformer, DERIVED_FEATURES

NUMERIC_FEATURE_COLS = NUMERIC_COLS + DERIVED_FEATURES


def criar_onehot_encoder():
    # Compatibilidade com versões que ainda usam o parâmetro sparse.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def construir_preprocessador(padronizar_numericas=False):
    etapas_num = [("imputador", SimpleImputer(strategy="median", add_indicator=True))]
    if padronizar_numericas:
        etapas_num.append(("escalador", StandardScaler(with_mean=False)))
    pipeline_num = Pipeline(etapas_num)

    pipeline_cat = Pipeline([
        ("imputador", SimpleImputer(strategy="constant", fill_value="Desconhecido")),
        ("onehot", criar_onehot_encoder()),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", pipeline_num, NUMERIC_FEATURE_COLS),
            ("cat", pipeline_cat, CATEGORICAL_COLS),
        ],
        remainder="drop",
    )


def construir_pipeline(modelo, padronizar_numericas=False):
    return Pipeline([
        ("features", FeatureEngineeringTransformer()),
        ("preprocessador", construir_preprocessador(padronizar_numericas)),
        ("modelo", modelo),
    ])

X_train_model = X_train_raw.drop(columns=[ID_COL]).copy()
X_test_model = X_test_raw.drop(columns=[ID_COL]).copy()

print(f"Treino: {X_train_model.shape}; teste reservado: {X_test_model.shape}")
print("A transformação será ajustada dentro de cada pipeline/fold; nenhuma matriz transformada será reutilizada entre treino e teste.")

## ETAPA 4A — MODELAGEM E TREINAMENTO

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, log_loss
)
from scipy.stats import ks_2samp
import shap

# Reserva-se uma validação interna a partir do treino. O teste de 25% permanece
# intocado até a avaliação final.
X_dev, X_valid, y_dev, y_valid = train_test_split(
    X_train_model,
    y_train,
    test_size=0.20,
    random_state=SEED + 10,
    stratify=y_train,
)
cv_treino = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

pipeline_logistica = construir_pipeline(
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"),
    padronizar_numericas=True,
)

pipeline_lgbm_base = construir_pipeline(
    LGBMClassifier(
        objective="binary",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    ),
    padronizar_numericas=False,
)

# Validação cruzada do baseline na subpartição de desenvolvimento.
cv_logistica = cross_validate(
    pipeline_logistica,
    X_dev,
    y_dev,
    cv=cv_treino,
    scoring={"roc_auc": "roc_auc", "average_precision": "average_precision"},
    n_jobs=1,
    return_train_score=False,
)
media_cv_logistica = float(np.mean(cv_logistica["test_roc_auc"]))

# Busca pequena, restrita à subpartição de desenvolvimento.
parametros_lgbm = {
    "modelo__n_estimators": [150, 250, 350],
    "modelo__learning_rate": [0.03, 0.05, 0.08],
    "modelo__num_leaves": [15, 31],
    "modelo__min_child_samples": [20, 40, 60],
    "modelo__reg_lambda": [0.0, 1.0, 5.0],
}
busca_lgbm = RandomizedSearchCV(
    estimator=pipeline_lgbm_base,
    param_distributions=parametros_lgbm,
    n_iter=5,
    scoring={"roc_auc": "roc_auc", "average_precision": "average_precision"},
    refit="roc_auc",
    cv=cv_treino,
    random_state=SEED,
    n_jobs=1,
    verbose=0,
)
busca_lgbm.fit(X_dev, y_dev)
media_cv_lgbm = float(busca_lgbm.best_score_)
pipeline_lgbm_otimizado = busca_lgbm.best_estimator_

resumo_cv = pd.DataFrame([
    {"modelo": "Regressão Logística", "roc_auc_cv_medio": media_cv_logistica,
     "roc_auc_cv_desvio": float(np.std(cv_logistica["test_roc_auc"])),
     "average_precision_cv_medio": float(np.mean(cv_logistica["test_average_precision"])),
     "criterio": "Baseline — folds da subpartição de desenvolvimento"},
    {"modelo": "LightGBM", "roc_auc_cv_medio": media_cv_lgbm,
     "roc_auc_cv_desvio": float(busca_lgbm.cv_results_["std_test_roc_auc"][busca_lgbm.best_index_]),
     "average_precision_cv_medio": float(busca_lgbm.cv_results_["mean_test_average_precision"][busca_lgbm.best_index_]),
     "criterio": "Melhor combinação da busca — folds da subpartição de desenvolvimento"},
])

# Ajusta cada candidato em desenvolvimento e compara numa validação separada.
pipeline_logistica.fit(X_dev, y_dev)
prob_valid_logistica = pipeline_logistica.predict_proba(X_valid)[:, 1]
prob_valid_lgbm = pipeline_lgbm_otimizado.predict_proba(X_valid)[:, 1]
resumo_validacao = pd.DataFrame([
    {"modelo": "Regressão Logística",
     "roc_auc_validacao": float(roc_auc_score(y_valid, prob_valid_logistica)),
     "average_precision_validacao": float(average_precision_score(y_valid, prob_valid_logistica))},
    {"modelo": "LightGBM",
     "roc_auc_validacao": float(roc_auc_score(y_valid, prob_valid_lgbm)),
     "average_precision_validacao": float(average_precision_score(y_valid, prob_valid_lgbm))},
]).sort_values("roc_auc_validacao", ascending=False)

# Seleção pelo ROC-AUC da validação interna separada, nunca pelo teste final.
nome_modelo_selecionado = resumo_validacao.iloc[0]["modelo"]
print("=== Validação cruzada na subpartição de desenvolvimento ===")
display(resumo_cv)
print("Melhores hiperparâmetros LightGBM:")
print(busca_lgbm.best_params_)
print("=== Comparação na validação interna separada ===")
display(resumo_validacao.round(4))
print(f"Modelo selecionado pela validação interna: {nome_modelo_selecionado}")

# Calibração de ambos os candidatos. O pré-processamento está dentro do estimador,
# portanto é reajustado corretamente em cada fold da calibração.
calibrado_logistica = CalibratedClassifierCV(
    estimator=pipeline_logistica,
    method="sigmoid",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    ensemble=True,
)
calibrado_lgbm = CalibratedClassifierCV(
    estimator=pipeline_lgbm_otimizado,
    method="sigmoid",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    ensemble=True,
)

calibrado_logistica.fit(X_train_model, y_train)
calibrado_lgbm.fit(X_train_model, y_train)
modelos_calibrados = {
    "Regressão Logística": calibrado_logistica,
    "LightGBM": calibrado_lgbm,
}
modelo_final = modelos_calibrados[nome_modelo_selecionado]


def calcular_metricas_binarias(y_true, probabilidade):
    probabilidade = np.clip(np.asarray(probabilidade), 1e-15, 1 - 1e-15)
    auc = roc_auc_score(y_true, probabilidade)
    ks = ks_2samp(probabilidade[np.asarray(y_true) == 1], probabilidade[np.asarray(y_true) == 0]).statistic
    return {
        "roc_auc": float(auc),
        "gini": float(2 * auc - 1),
        "ks": float(ks),
        "average_precision": float(average_precision_score(y_true, probabilidade)),
        "brier_score": float(brier_score_loss(y_true, probabilidade)),
        "log_loss": float(log_loss(y_true, probabilidade)),
    }

probabilidades_teste = {}
metricas_teste = {}
linhas_comparacao = []
for nome, modelo in modelos_calibrados.items():
    prob = modelo.predict_proba(X_test_model)[:, 1]
    probabilidades_teste[nome] = prob
    metricas = calcular_metricas_binarias(y_test, prob)
    metricas_teste[nome] = metricas
    linhas_comparacao.append({
        "modelo": nome,
        "roc_auc_validacao": float(resumo_validacao.loc[resumo_validacao["modelo"] == nome, "roc_auc_validacao"].iloc[0]),
        **metricas,
    })

comparacao_modelos = pd.DataFrame(linhas_comparacao).sort_values("roc_auc_validacao", ascending=False)
prob_final_teste = probabilidades_teste[nome_modelo_selecionado]
metricas_final_teste = metricas_teste[nome_modelo_selecionado]

print("=== Comparação final na base de teste (holdout) ===")
display(comparacao_modelos.round(4))
print("Modelo final:", nome_modelo_selecionado)
print("Métricas finais:", {k: round(v, 4) for k, v in metricas_final_teste.items()})

# Tabela de risco por decil: D10 representa o maior risco.
df_avaliacao = pd.DataFrame({"y_true": np.asarray(y_test), "pd_prevista": prob_final_teste}, index=y_test.index)
df_avaliacao["decil"] = pd.qcut(
    df_avaliacao["pd_prevista"].rank(method="first"), q=10, labels=False, duplicates="drop"
) + 1

tabela_decis = (
    df_avaliacao.groupby("decil", observed=True)
    .agg(registros=("y_true", "size"), defaults=("y_true", "sum"),
         default_observado=("y_true", "mean"), pd_media=("pd_prevista", "mean"))
    .reset_index().sort_values("decil", ascending=False)
)
tabela_decis["default_observado_pct"] = 100 * tabela_decis["default_observado"]
tabela_decis["pd_media_pct"] = 100 * tabela_decis["pd_media"]

# Curvas de calibração por quantis calculadas uma única vez para ambos os candidatos.
curvas_calibracao = {}
for nome, prob in probabilidades_teste.items():
    fracao_positivos, media_predita = calibration_curve(
        y_test, prob, n_bins=10, strategy="quantile"
    )
    curvas_calibracao[nome] = pd.DataFrame({
        "pd_media_predita": media_predita,
        "default_observado": fracao_positivos,
    })
tabela_calibracao = curvas_calibracao[nome_modelo_selecionado]

# SHAP calculado uma única vez; a etapa 4B apenas plota os objetos resultantes.
shap_resultado = None
shap_status = "Não executado"
try:
    componente_calibrado = modelo_final.calibrated_classifiers_[0]
    estimador_base = getattr(componente_calibrado, "estimator", None)
    if estimador_base is None:
        estimador_base = getattr(componente_calibrado, "base_estimator", None)
    if estimador_base is None:
        raise AttributeError("Não foi possível acessar o estimador-base do calibrador.")

    # Inclui a observação de maior PD no início para que haja uma explicação local útil.
    indice_maior_risco = df_avaliacao["pd_prevista"].idxmax()
    X_shap_raw = X_test_model.loc[[indice_maior_risco]].copy()
    amostra_adicional = X_test_model.drop(index=indice_maior_risco).sample(
        n=min(199, len(X_test_model) - 1), random_state=SEED
    )
    X_shap_raw = pd.concat([X_shap_raw, amostra_adicional])

    transformador_completo = estimador_base[:-1]
    matriz_shap = transformador_completo.transform(X_shap_raw)
    if hasattr(matriz_shap, "toarray"):
        matriz_shap = matriz_shap.toarray()
    nomes_features_shap = estimador_base.named_steps["preprocessador"].get_feature_names_out()
    nomes_features_shap = [n.split("__", 1)[-1] for n in nomes_features_shap]
    X_shap_df = pd.DataFrame(matriz_shap, columns=nomes_features_shap, index=X_shap_raw.index)

    estimador_modelo_base = estimador_base.named_steps["modelo"]
    if isinstance(estimador_modelo_base, LGBMClassifier):
        explainer_shap = shap.TreeExplainer(estimador_modelo_base)
    else:
        background = X_shap_df.iloc[1:min(51, len(X_shap_df))]
        explainer_shap = shap.LinearExplainer(estimador_modelo_base, background)
    shap_resultado = explainer_shap(X_shap_df)

    shap_array = np.asarray(shap_resultado.values)
    if shap_array.ndim == 3:
        shap_array = shap_array[:, :, 1]
    shap_base = np.asarray(shap_resultado.base_values)
    if shap_base.ndim == 2:
        shap_base = shap_base[:, 1]
    shap_resultado_binario = shap.Explanation(
        values=shap_array,
        base_values=shap_base,
        data=X_shap_df.to_numpy(),
        feature_names=nomes_features_shap,
    )
    importancia_shap = pd.DataFrame({
        "variavel": nomes_features_shap,
        "mean_abs_shap": np.abs(shap_array).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)
    escala_shap = (
        "log-odds do estimador-base"
        if isinstance(estimador_modelo_base, LogisticRegression)
        else "output bruto do estimador-base"
    )
    shap_status = (
        f"SHAP calculado para {type(estimador_modelo_base).__name__}; "
        f"contribuições na escala de {escala_shap}, após pré-processamento."
    )
except Exception as exc:
    importancia_shap = pd.DataFrame(columns=["variavel", "mean_abs_shap"])
    shap_status = f"SHAP não pôde ser calculado nesta execução: {type(exc).__name__}: {exc}"

print("=== Esperado versus observado por decil (D10 = maior risco) ===")
display(tabela_decis[["decil", "registros", "defaults", "pd_media_pct", "default_observado_pct"]].round(2))
print("=== Status da explicabilidade ===")
print(shap_status)
if not importancia_shap.empty:
    display(importancia_shap.head(12))

## ETAPA 4B — VISUALIZAÇÃO DO MODELO

In [ ]:
# 1. Discriminação (ROC-AUC no teste)
plt.figure(figsize=(6, 4))
nomes = comparacao_modelos["modelo"].tolist()
valores_auc = comparacao_modelos["roc_auc"].tolist()
plt.bar(nomes, valores_auc, color=["#2ca02c", "#d62728"])
plt.ylim(0.5, 1.0)
plt.title("ROC-AUC no teste")
plt.xticks(rotation=15)
for i, v in enumerate(valores_auc):
    plt.text(i, v + 0.008, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.savefig(PASTA_RELATORIOS / "avaliacao_roc_auc.png", dpi=160, bbox_inches="tight")
plt.show()

# 2. Curva de Calibração
plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibração perfeita")
for nome, curva in curvas_calibracao.items():
    plt.plot(
        curva["pd_media_predita"], curva["default_observado"],
        marker="o", label=nome
    )
plt.title("Curva de calibração")
plt.xlabel("PD média prevista")
plt.ylabel("Default observado")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig(PASTA_RELATORIOS / "avaliacao_calibracao.png", dpi=160, bbox_inches="tight")
plt.show()

# 3. Ordenação por decil (D10 maior risco)
plt.figure(figsize=(8, 4))
tabela_plot = tabela_decis.sort_values("decil", ascending=False)
x = np.arange(len(tabela_plot))
width = 0.38
plt.bar(x - width/2, tabela_plot["pd_media_pct"], width, label="PD prevista")
plt.bar(x + width/2, tabela_plot["default_observado_pct"], width, label="Default observado")
plt.title("Ordenação por decil — D10 maior risco")
plt.xlabel("Decil")
plt.ylabel("Percentual (%)")
plt.xticks(x, [f"D{int(v)}" for v in tabela_plot["decil"]])
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig(PASTA_RELATORIOS / "avaliacao_decis.png", dpi=160, bbox_inches="tight")
plt.show()

# SHAP global e local
if shap_resultado is not None:
    plt.figure()
    shap.summary_plot(shap_resultado_binario.values, X_shap_df, max_display=12, show=False)
    plt.title(f"SHAP global — contribuições em {escala_shap}")
    plt.tight_layout()
    plt.savefig(PASTA_RELATORIOS / "shap_global.png", dpi=160, bbox_inches="tight")
    plt.show()

    plt.figure()
    shap.plots.waterfall(shap_resultado_binario[0], max_display=12, show=False)
    plt.title(f"SHAP local — maior PD no teste ({escala_shap})")
    plt.tight_layout()
    plt.savefig(PASTA_RELATORIOS / "shap_waterfall_local.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print(shap_status)


## ETAPA 5A — MONITORAMENTO DA ESTABILIDADE (PSI/CSI)

In [ ]:
from feature_engineering import FeatureEngineeringTransformer


def calcular_psi_scores(referencia, atual, bins=10, epsilon=1e-6):
    referencia = pd.Series(referencia).dropna().astype(float).to_numpy()
    atual = pd.Series(atual).dropna().astype(float).to_numpy()
    if len(referencia) == 0 or len(atual) == 0:
        return np.nan

    quantis = np.unique(np.quantile(referencia, np.linspace(0, 1, bins + 1)))
    cortes = np.unique(np.r_[-np.inf, quantis[1:-1], np.inf])
    if len(cortes) < 2:
        cortes = np.array([-np.inf, np.inf])

    pct_ref = np.histogram(referencia, bins=cortes)[0] / len(referencia)
    pct_atual = np.histogram(atual, bins=cortes)[0] / len(atual)
    pct_ref = (pct_ref + epsilon) / (pct_ref.sum() + epsilon * len(pct_ref))
    pct_atual = (pct_atual + epsilon) / (pct_atual.sum() + epsilon * len(pct_atual))
    return float(np.sum((pct_atual - pct_ref) * np.log(pct_atual / pct_ref)))


def calcular_csi_variavel(referencia, atual, bins=10, epsilon=1e-6):
    ref = pd.Series(referencia).reset_index(drop=True)
    cur = pd.Series(atual).reset_index(drop=True)
    numerica = pd.api.types.is_numeric_dtype(ref) and pd.api.types.is_numeric_dtype(cur)

    if numerica:
        ref_num = pd.to_numeric(ref, errors="coerce")
        cur_num = pd.to_numeric(cur, errors="coerce")
        valores_ref = ref_num.dropna().to_numpy(dtype=float)
        if len(valores_ref) == 0:
            categorias = ["__AUSENTE__"]
            pct_ref = np.array([1.0])
            pct_cur = np.array([float(cur_num.isna().mean())])
        else:
            quantis = np.unique(np.quantile(valores_ref, np.linspace(0, 1, bins + 1)))
            cortes = np.unique(np.r_[-np.inf, quantis[1:-1], np.inf])
            if len(cortes) < 2:
                cortes = np.array([-np.inf, np.inf])
            hist_ref = np.histogram(ref_num.dropna(), bins=cortes)[0]
            hist_cur = np.histogram(cur_num.dropna(), bins=cortes)[0]
            pct_ref = np.r_[hist_ref / max(len(ref_num), 1), ref_num.isna().mean()]
            pct_cur = np.r_[hist_cur / max(len(cur_num), 1), cur_num.isna().mean()]
    else:
        ref_cat = ref.astype("string").fillna("__AUSENTE__").astype(str)
        cur_cat = cur.astype("string").fillna("__AUSENTE__").astype(str)
        categorias = sorted(set(ref_cat.unique()) | set(cur_cat.unique()))
        pct_ref = ref_cat.value_counts(normalize=True).reindex(categorias, fill_value=0).to_numpy()
        pct_cur = cur_cat.value_counts(normalize=True).reindex(categorias, fill_value=0).to_numpy()

    pct_ref = np.asarray(pct_ref, dtype=float)
    pct_cur = np.asarray(pct_cur, dtype=float)
    pct_ref = (pct_ref + epsilon) / (pct_ref.sum() + epsilon * len(pct_ref))
    pct_cur = (pct_cur + epsilon) / (pct_cur.sum() + epsilon * len(pct_cur))
    return float(np.sum((pct_cur - pct_ref) * np.log(pct_cur / pct_ref)))


def classificar_indice(valor):
    if pd.isna(valor):
        return "Indisponível"
    if valor < 0.10:
        return "Verde — estável"
    if valor <= 0.25:
        return "Amarelo — atenção"
    return "Vermelho — revisão"


# Cenários: referência, reamostragem estável e crise. Tudo permanece em escala original.
X_referencia_raw = X_test_model.copy()
X_safra_estavel_raw = X_referencia_raw.sample(
    n=len(X_referencia_raw), replace=True, random_state=SEED + 1
).reset_index(drop=True)
X_safra_crise_raw = X_referencia_raw.sample(
    n=len(X_referencia_raw), replace=True, random_state=SEED + 2
).reset_index(drop=True)

rng = np.random.default_rng(SEED + 3)
mask_atrasos = rng.random(len(X_safra_crise_raw)) < 0.60
mask_renda = rng.random(len(X_safra_crise_raw)) < 0.40
X_safra_crise_raw.loc[mask_atrasos, "pagamentos_atrasados_ultimos_12meses"] = (
    X_safra_crise_raw.loc[mask_atrasos, "pagamentos_atrasados_ultimos_12meses"] + 2
)
X_safra_crise_raw.loc[mask_renda, "renda_mensal"] = (
    X_safra_crise_raw.loc[mask_renda, "renda_mensal"] * 0.80
)

# predict_proba recebe dados brutos; FeatureEngineeringTransformer e preprocessor
# recalculam automaticamente as variáveis derivadas e as transformações.
score_referencia = modelo_final.predict_proba(X_referencia_raw)[:, 1]
score_safra_estavel = modelo_final.predict_proba(X_safra_estavel_raw)[:, 1]
score_safra_crise = modelo_final.predict_proba(X_safra_crise_raw)[:, 1]

psi_estavel = calcular_psi_scores(score_referencia, score_safra_estavel)
psi_crise = calcular_psi_scores(score_referencia, score_safra_crise)
resumo_psi = pd.DataFrame([
    {"cenario": "Safra estável", "psi_score": psi_estavel, "status": classificar_indice(psi_estavel)},
    {"cenario": "Safra de crise simulada", "psi_score": psi_crise, "status": classificar_indice(psi_crise)},
])

# CSI em atributos originais e derivados. A engenharia é determinística e não ajusta parâmetros.
engenharia = FeatureEngineeringTransformer().fit(X_referencia_raw)
features_referencia = engenharia.transform(X_referencia_raw)
features_crise = engenharia.transform(X_safra_crise_raw)
colunas_csi = [c for c in features_referencia.columns if c != ID_COL]
linhas_csi = []
for coluna in colunas_csi:
    valor_csi = calcular_csi_variavel(features_referencia[coluna], features_crise[coluna])
    linhas_csi.append({"variavel": coluna, "csi": valor_csi, "status": classificar_indice(valor_csi)})
resultado_csi = pd.DataFrame(linhas_csi).sort_values("csi", ascending=False).reset_index(drop=True)

print("=== PSI do score ===")
display(resumo_psi.round(4))
print("=== CSI por característica — maiores alterações primeiro ===")
display(resultado_csi.head(15).round(4))
print("Nota: cenário de crise é artificial e demonstra sensibilidade distributiva, não perda comprovada de desempenho.")

## ETAPA 5B — VISUALIZAÇÃO DO VIGIA

In [ ]:
# 1. Distribuição de PD por cenário (Estabilidade/PSI)
plt.figure(figsize=(7, 5))
plt.hist(score_referencia, bins=30, density=True, histtype="step", linewidth=2, label="Referência (teste)")
plt.hist(score_safra_estavel, bins=30, density=True, histtype="step", linewidth=2, label=f"Estável — PSI {psi_estavel:.3f}")
plt.hist(score_safra_crise, bins=30, density=True, histtype="step", linewidth=2, label=f"Crise — PSI {psi_crise:.3f}")
plt.title("Distribuição de PD por cenário")
plt.xlabel("Probabilidade estimada de default")
plt.ylabel("Densidade")
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig(PASTA_RELATORIOS / "vigia_distribuicao_pd_psi.png", dpi=160, bbox_inches="tight")
plt.show()

# 2. Características com maior CSI (Desvio Distributivo)
plt.figure(figsize=(8, 5))
top_csi = resultado_csi.head(12).sort_values("csi", ascending=True)
plt.barh(top_csi["variavel"], top_csi["csi"])
plt.axvline(0.10, color="orange", linestyle="--", label="Atenção (0,10)")
plt.axvline(0.25, color="red", linestyle=":", label="Revisão (0,25)")
plt.title("Características com maior CSI (Population Stability Index)")
plt.xlabel("Valor do CSI")
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig(PASTA_RELATORIOS / "vigia_caracteristicas_csi.png", dpi=160, bbox_inches="tight")
plt.show()


## ETAPA 6 — SALVAMENTO DOS ARTEFATOS E SCORING DA BASE

In [ ]:
import joblib
import sklearn
import lightgbm

# Score de todos os registros: entrada bruta; o pipeline executa a engenharia internamente.
X_todos_model = df_ingested.drop(columns=[TARGET_COL, ID_COL]).copy()
pd_todos = modelo_final.predict_proba(X_todos_model)[:, 1]

# Faixas relativas à distribuição pontuada nesta base; não são limites normativos de crédito.
rank_percentil = pd.Series(pd_todos).rank(method="first", pct=True)
faixa_relativa = pd.cut(
    rank_percentil,
    bins=[0, 0.70, 0.90, 1.0],
    labels=["Baixo", "Médio", "Alto"],
    include_lowest=True,
)
particao = pd.Series("não classificado", index=df_ingested.index, dtype="object")
particao.loc[X_train_raw.index] = "treino"
particao.loc[X_test_raw.index] = "teste"

arquivo_scores = pd.DataFrame({
    ID_COL: df_ingested[ID_COL].values,
    "probabilidade_default": pd_todos,
    "faixa_risco_relativa": faixa_relativa.astype(str).to_numpy(),
    "particao": particao.reindex(df_ingested.index).to_numpy(),
})

# Arquivos tabulares e matrizes utilizadas no trabalho.
resumo_estrutura.to_csv(PASTA_RELATORIOS / "resumo_estrutura.csv", index=False)
resumo_nulos.to_csv(PASTA_RELATORIOS / "valores_ausentes.csv")
relatorio_regras.to_csv(PASTA_RELATORIOS / "regras_qualidade.csv", index=False)
resumo_numerico_treino.to_csv(PASTA_RELATORIOS / "estatisticas_numericas_treino.csv")
matriz_correlacao.to_csv(PASTA_RELATORIOS / "matriz_correlacao_treino.csv")
indice_default_atrasos.to_csv(PASTA_RELATORIOS / "default_por_atrasos.csv", index=False)
indice_default_uso.to_csv(PASTA_RELATORIOS / "default_por_uso_credito.csv", index=False)
resumo_cv.to_csv(PASTA_RELATORIOS / "comparacao_validacao_cruzada.csv", index=False)
comparacao_modelos.to_csv(PASTA_RELATORIOS / "comparacao_modelos_teste.csv", index=False)
resumo_validacao.to_csv(PASTA_RELATORIOS / "comparacao_validacao_interna.csv", index=False)
tabela_decis.to_csv(PASTA_RELATORIOS / "ordenacao_risco_decis.csv", index=False)
tabela_calibracao.to_csv(PASTA_RELATORIOS / "curva_calibracao.csv", index=False)
resultado_csi.to_csv(PASTA_RELATORIOS / "csi_caracteristicas.csv", index=False)
importancia_shap.to_csv(PASTA_RELATORIOS / "importancia_shap.csv", index=False)
arquivo_scores.to_csv(PASTA_RELATORIOS / "scores_base_completa.csv", index=False)
indice_default_categorias.to_csv(PASTA_RELATORIOS / "default_por_categoria.csv", index=False)

# Salva o modelo calibrado selecionado. A classe customizada está no módulo auxiliar.
caminho_modelo = PASTA_ARTEFATOS / "modelo_final_calibrado.joblib"
joblib.dump(modelo_final, caminho_modelo)

metadados = {
    "versao_pipeline": "2.0",
    "modelo_selecionado": nome_modelo_selecionado,
    "criterio_selecao": "ROC-AUC em validação interna separada da subpartição de desenvolvimento; teste final não participa da escolha",
    "resultados_validacao_interna": resumo_validacao.to_dict(orient="records"),
    "melhores_parametros_lightgbm": busca_lgbm.best_params_,
    "semente": SEED,
    "arquivo_origem": str(CAMINHO_ENTRADA_CSV),
    "n_registros": int(len(df_ingested)),
    "n_treino": int(len(X_train_model)),
    "n_teste": int(len(X_test_model)),
    "taxa_default_total": float(y_full.mean()),
    "metricas_teste_modelo_selecionado": metricas_final_teste,
    "psi": {"safra_estavel": float(psi_estavel), "safra_crise_simulada": float(psi_crise)},
    "shap_status": shap_status,
    "versoes": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "lightgbm": lightgbm.__version__,
        "shap": shap.__version__,
    },
    "observacao": "A simulação de crise avalia estabilidade distributiva; não mede degradação real sem rótulos futuros.",
}
(PASTA_ARTEFATOS / "metadados_modelo.json").write_text(
    json.dumps(metadados, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
)

print("=== Entregáveis gravados ===")
print(f"Modelo: {caminho_modelo}")
print(f"Metadados: {PASTA_ARTEFATOS / 'metadados_modelo.json'}")
print(f"Scores da base: {PASTA_RELATORIOS / 'scores_base_completa.csv'}")
print(f"Relatórios e gráficos: {PASTA_RELATORIOS}")
print(f"Total de arquivos de relatório: {len(list(PASTA_RELATORIOS.iterdir()))}")